In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack
import fitsio
import scipy.spatial.transform as trans
from desimodel.footprint import is_point_in_desi
import astropy.io.ascii
from scipy.spatial import cKDTree

In [2]:
#I need to use the check if point is in DESI footprint
#and the ebits & imbits mask
def generateHoleRands(rantab, tiletab,ralim,declim,outfilename):
    #check if point is in DESI footprint
    #I think this is redundant with the next step
    sel1 = (rantab['RA'] > ralim[0])&(rantab['RA'] < ralim[1])&(rantab['DEC']>declim[0])&(rantab['DEC'] < declim[1])
    rantab = rantab[sel1]
    print("initial number of randoms, subject to angular cut: "+str(len(rantab)))
    sel2 = is_point_in_desi(tiletab,rantab['RA'],rantab['DEC'])
    rantab = rantab[sel2]
    print("randoms after DESI footprint applied: "+str(len(rantab)))
    sel3 = (rantab['MASKBITS'] & 0x1007 == 0)
    rantab = rantab[sel3]
    print("randoms after imbits and ebits applied: "+str(len(rantab)))
    astropy.io.ascii.write(rantab, 'randoms/'+outfilename+'.csv', overwrite=True,format='csv')

In [3]:
tiletab = Table.read('/global/cfs/cdirs/desi/survey/catalogs/Y1/LSS/tiles-DARK.fits')
randir = "/global/cfs/cdirs/desi/target/catalogs/dr9/0.49.0/randoms/resolve/"
ralim = [130,223]
declim  = [-5.25,3.75]

In [5]:
for i in range(18):
    outfilename = "ranEbits"+str(i)
    ranfile = randir + "randoms-1-"+str(i)+".fits"
    rantab = Table(fitsio.read(ranfile,columns=['RA','DEC','MASKBITS']))
    generateHoleRands(rantab, tiletab, ralim, declim,outfilename)

initial number of randoms, subject to angular cut: 2089380
randoms after DESI footprint applied: 2089380
randoms after imbits and ebits applied: 2067803


KeyboardInterrupt: 

In [4]:
rosettes = {'RA': [150.100, 179.600, 183.100, 189.900, 194.750, 210.000, 215.500, 217.800, 216.300, 219.800, 218.050, 
             242.750, 241.050, 245.880, 252.500, 269.730, 194.750, 212.800, 269.730, 236.100],
            'DEC': [2.182, 0.000, 0.000, 61.800, 28.200, 5.000, 52.500, 34.400, 0.600, 0.600, 2.430, 54.980, 43.450, 43.450,
              34.500, 66.020, 24.700, 0.600, 62.520, 43.450],
            'ID': np.array(np.arange(20)+1)}

def checkAngularSep(phi1,phi2,theta1,theta2,psimin,psimax):
    #arguments: RA1, RA2, Dec1, Dec2, min separation, max separation (in DEGREES)
    theta1 = theta1*np.pi/180
    theta2 = theta2*np.pi/180
    phi1 = phi1*np.pi/180
    phi2 = phi2*np.pi/180
    psimin = psimin*np.pi/180
    psimax = psimax*np.pi/180
    cospsi = np.sin(theta1)*np.sin(theta2)+np.cos(theta1)*np.cos(theta2)*np.cos(phi1-phi2)
    return (cospsi**2 > np.cos(psimax)**2)&(cospsi**2 < np.cos(psimin)**2)

def angularCut(table,ramin,ramax,decmin,decmax):
    return (table['DEC'] > decmin) & (table['DEC'] < decmax) & \
            (table['RA'] > ramin) & (table['RA'] < ramax)

def rosetteOR(table, psimin, psimax, rosettes, ra, dec):
    rosettemask = [checkAngularSep(rosettes['RA'][i],table[ra],rosettes['DEC'][i],table[dec],psimin,psimax)
                   for i in range(len(rosettes['RA']))]
    #return true if it belongs to any of the rosettes
    table['rosette'] = [rosettes['ID'][r] for r in np.transpose(rosettemask)]
    #store boolean mask of all rosettes it belongs to (later we will convert this to a rosette index)
    return np.any(rosettemask,axis=0)

def raDecToUnitSphere(a, d):
    a = a*np.pi/180
    d = d*np.pi/180
    x = np.cos(a)*np.cos(d)
    y = np.sin(a)*np.cos(d)
    z = np.sin(d)
    return [np.array([x[i],y[i],z[i],1]) for i in range(len(a))]

def MCintegrateholes(randtable,holetable,mask,maskargs,R_hole):
    rands = randtable[mask(randtable,*maskargs)]
    rands['UScoord'] = raDecToUnitSphere(rands['RA'],rands['DEC'])
    randUSmap = [coord[:3] for coord in rands['UScoord']]
    randtree_US = cKDTree(randUSmap)

    
    holemap = [[holetable['x'][i],holetable['y'][i],holetable['z'][i]] for i in range(len(holetable))]
    holetree = cKDTree(holemap)

    holeneighbors = randtree_US.query_ball_tree(holetree,R_hole)
    holemask = [len(n) < 1 for n in holeneighbors]
    randcleaned = rands[holemask]

    return len(randcleaned)/len(rands)

In [5]:
randir = "/global/cfs/cdirs/desi/target/catalogs/dr9/0.49.0/randoms/resolve/"
rantables = []
for i in range(1):
    ranfile = randir + "randoms-1-"+str(i)+".fits"
    rantab = Table(fitsio.read(ranfile,columns=['RA','DEC','MASKBITS']))
    rantables.append(rantab)
rantable = vstack(rantables)
rantable['RA'] = rantable['RA']+1

In [5]:
#loadholes
holetableselg = []
holetableslrg = []
for i in range(240):
    spelg = Table.read("datafiles/forFAelg_tile"+str(i)+"_holes.csv",format = "csv")
    splrg = Table.read("datafiles/forFAlrg_tile"+str(i)+"_holes.csv",format = "csv")
    holetableselg.append(spelg)
    holetableslrg.append(splrg)
holetableelg = vstack(holetableselg)
holetablelrg = vstack(holetableslrg)
print(len(holetableelg),len(holetablelrg))

2218580 2349008


In [6]:
ralim = [130,223]
declim  = [-5.25,3.75]

maskargs = [*ralim,*declim]

mask = angularCut
R_hole = 0.03*np.pi/180 #convert from degrees to radians

elgpass = MCintegrateholes(rantable,holetableelg,mask,maskargs,R_hole)
lrgpass = MCintegrateholes(rantable,holetablelrg,mask,maskargs,R_hole)

In [7]:
print(elgpass)
print(lrgpass) #0.867, 0.878

0.8659814443284376
0.8772861289150519


In [6]:
#area enlosed by ra,dec limits
ralimrad = np.pi/180*np.array(ralim)
declimrad = np.pi/180*np.array(declim)

area = (np.sin(declimrad[1])-np.sin(declimrad[0]))*(ralimrad[1]-ralimrad[0])
print(area)
print(area/np.pi**2*180**2)
print((130-223)*(-5.25-3.75))

#area enclosed by rosette limits

0.25468091503612245
836.0681251073842
837.0


In [8]:
#loadholes
holetableselg = []
holetableslrg = []
for i in range(117):
    spelg = Table.read("datafiles/Holes_ELG_SV3_tile"+str(i)+"_holes.csv",format = "csv")
    splrg = Table.read("datafiles/Holes_LRG_SV3_tile"+str(i)+"_holes.csv",format = "csv")
    holetableselg.append(spelg)
    holetableslrg.append(splrg)
holetableelg = vstack(holetableselg)
holetablelrg = vstack(holetableslrg)
print(len(holetableelg),len(holetablelrg))

4646361 4697603


In [9]:
psilim = [0.2,1.5]

maskargs = [*psilim, rosettes,'RA','DEC']

mask = rosetteOR
R_hole = R_hole = 0.03*np.pi/180 #convert from degrees to radians

elgpass = MCintegrateholes(rantable,holetableelg,mask,maskargs,R_hole)
lrgpass = MCintegrateholes(rantable,holetablelrg,mask,maskargs,R_hole)

In [10]:
print(elgpass)
print(lrgpass)

0.9067233610305354
0.8893598303401511


In [19]:
#area enclosed by rosette limits
psilimrad = np.pi/180*np.array(psilim)

area=2*np.pi*(np.cos(psilimrad[0])-np.cos(psilimrad[1]))*20
print(area)
print(area*(180/np.pi)**2)

0.042296227273166336
138.85032347389037
